In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from prophet import Prophet
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_components
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Directories
input_dir = "../Data/Prophet_Preprocessed"
output_dir = "prophet_results"
os.makedirs(output_dir, exist_ok=True)

# Search grids
cp_grid = [0.05, 0.1, 0.2, 0.5]
sp_grid = [1, 5, 10, 20]

# Results summary
results = []

# Run for each file (stock)
for file in tqdm(os.listdir(input_dir), desc="Running Prophet Grid Search"):
    if file.endswith(".csv"):
        stock = file.replace(".csv", "")
        df = pd.read_csv(os.path.join(input_dir, file))
        df.rename(columns={'Date': 'ds', 'Close': 'y'}, inplace=True)

        best_rmse = float("inf")
        best_model = None
        best_params = {}

        # Try all param combinations
        for cp in cp_grid:
            for sp in sp_grid:
                try:
                    model = Prophet(
                        changepoint_prior_scale=cp,
                        seasonality_prior_scale=sp,
                        seasonality_mode='multiplicative',
                        yearly_seasonality=True,
                        weekly_seasonality=False,
                        daily_seasonality=False
                    )
                    model.add_seasonality(name='monthly', period=30.5, fourier_order=5)
                    model.add_country_holidays(country_name='US')
                    model.fit(df)

                    # Cross-validation
                    df_cv = cross_validation(model, initial='1095 days', period='180 days', horizon='365 days')
                    df_p = performance_metrics(df_cv, rolling_window=1)

                    rmse = df_p['rmse'].mean()
                    mae = df_p['mae'].mean()

                    if rmse < best_rmse:
                        best_rmse = rmse
                        best_model = model
                        best_df_p = df_p
                        best_params = {
                            'changepoint_prior_scale': cp,
                            'seasonality_prior_scale': sp,
                            'rmse': rmse,
                            'mae': mae
                        }

                except Exception as e:
                    print(f"Error for {stock} with cp={cp}, sp={sp}: {e}")

        if best_model:
            # Forecast
            future = best_model.make_future_dataframe(periods=365)
            forecast = best_model.predict(future)

            # Save forecast and metrics
            forecast.to_csv(os.path.join(output_dir, f"{stock}_forecast.csv"), index=False)
            best_df_p.to_csv(os.path.join(output_dir, f"{stock}_cv_metrics.csv"), index=False)

            # Save plot
            fig = best_model.plot(forecast)
            fig.savefig(os.path.join(output_dir, f"{stock}_forecast.png"))
            plt.close(fig)

            # Add to summary
            results.append({
                'Stock': stock,
                'RMSE': best_params['rmse'],
                'MAE': best_params['mae'],
                'changepoint_prior_scale': best_params['changepoint_prior_scale'],
                'seasonality_prior_scale': best_params['seasonality_prior_scale']
            })

# Final summary
pd.DataFrame(results).to_csv(os.path.join(output_dir, "summary.csv"), index=False)
print("✅ Grid search + forecasts saved in prophet_results/")


Running Prophet Grid Search:   0%|                                                               | 0/6 [00:00<?, ?it/s]15:39:38 - cmdstanpy - INFO - Chain [1] start processing
15:39:40 - cmdstanpy - INFO - Chain [1] done processing

  0%|                                                                                           | 0/33 [00:00<?, ?it/s]15:39:40 - cmdstanpy - INFO - Chain [1] start processing
15:39:41 - cmdstanpy - INFO - Chain [1] done processing

  3%|██▌                                                                                | 1/33 [00:00<00:22,  1.40it/s]15:39:41 - cmdstanpy - INFO - Chain [1] start processing
15:39:42 - cmdstanpy - INFO - Chain [1] done processing

  6%|█████                                                                              | 2/33 [00:01<00:24,  1.26it/s]15:39:42 - cmdstanpy - INFO - Chain [1] start processing
15:39:42 - cmdstanpy - INFO - Chain [1] done processing

  9%|███████▌                                                       